# Probe-Based Influence Analysis

This notebook:
1. Extracts/trains a linear probe on the final residual stream for a target token (e.g., " frog")
2. Uses the probe direction as the measurement function in Kronfluence
3. Finds the most influential training documents for that direction

The key insight: instead of using a measurement dataset with contrastive loss, we directly measure
how much the final residual stream aligns with the probe direction.

In [ ]:
import os
import sys
import json
import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, GPTNeoForCausalLM, default_data_collator
from huggingface_hub import hf_hub_download, repo_exists, login
from dotenv import load_dotenv

# Configuration
cfg_param = "8M"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 3407
max_length = 256

# Set seeds
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

# HuggingFace setup
load_dotenv()
hf_token = os.getenv('HF_TOKEN')
if hf_token:
    login(token=hf_token)
    print("Logged in to HuggingFace")

HF_USERNAME = os.getenv('HF_USERNAME', 'jrosseruk')
HF_REPO_PREFIX = f"{HF_USERNAME}/gpt-tinystories"

In [ ]:
# Apply kronfluence patches
from infusion.kronfluence_patches import apply_patches
apply_patches()

# Kronfluence imports
sys.path.append("")
sys.path.append("kronfluence")
sys.path.append("kronfluence/kronfluence")

from kronfluence.analyzer import Analyzer, prepare_model
from kronfluence.arguments import FactorArguments, ScoreArguments
from kronfluence.task import Task
from kronfluence.utils.dataset import DataLoaderKwargs
from kronfluence.utils.common.factor_arguments import extreme_reduce_memory_factor_arguments

## Configuration

In [ ]:
# Checkpoint to analyze
checkpoint = 292000

# Target token for probe direction
target_word = " frog"

# Probe type: 'unembedding' (use lm_head weights) or 'trained' (train a logistic probe)
probe_type = 'unembedding'

# Damping factor for influence scores
damping = 1e-8

print(f"Target word: '{target_word}'")
print(f"Probe type: {probe_type}")
print(f"Checkpoint: {checkpoint}")

## Load Model and Data

In [ ]:
def load_model_for_inference(checkpoint_step, device='cuda'):
    """Load a trained model from HuggingFace."""
    repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    subfolder = f"checkpoint-{checkpoint_step}"
    print(f"Loading model from {repo_name}/{subfolder}...")
    
    model = GPTNeoForCausalLM.from_pretrained(repo_name, subfolder=subfolder)
    tokenizer = AutoTokenizer.from_pretrained(f"roneneldan/TinyStories-{cfg_param}")
    tokenizer.pad_token = tokenizer.eos_token
    
    model = model.to(device)
    model.eval()
    print(f"Model loaded! Hidden size: {model.config.hidden_size}")
    return model, tokenizer


def load_checkpoint_data(checkpoint_step):
    """Load training/validation data from a specific checkpoint."""
    repo_name = f"{HF_REPO_PREFIX}-{cfg_param}"
    data_path = hf_hub_download(
        repo_id=repo_name, 
        filename=f'checkpoint-{checkpoint_step}/data_tracker.json'
    )
    with open(data_path, 'r') as f:
        data_tracker = json.load(f)
    
    print(f"Loaded data: {len(data_tracker['train_data'])} train, {len(data_tracker['val_data'])} val")
    return data_tracker


# Load model and data
model, tokenizer = load_model_for_inference(checkpoint)
data = load_checkpoint_data(checkpoint)

# Get target token ID
target_token_ids = tokenizer.encode(target_word, add_special_tokens=False)
if len(target_token_ids) != 1:
    print(f"Warning: '{target_word}' tokenizes to {len(target_token_ids)} tokens. Using first.")
target_token_id = target_token_ids[0]
print(f"Target token: '{target_word}' -> ID {target_token_id}")

## Extract Probe Direction

Two options:
1. **Unembedding direction**: Use the column of the LM head (unembedding matrix) for the target token. This is the direction that, when the residual stream aligns with it, produces high logit for that token.

2. **Trained probe**: Train a logistic regression probe on (residual, is_next_token_target) pairs.

In [ ]:
def get_unembedding_direction(model, token_id):
    """
    Get the unembedding direction for a token.
    
    For GPT-Neo: logit_i = (LayerNorm(residual) @ lm_head.weight.T)[i]
    The 'direction' for token i is lm_head.weight[i, :] (row of weight matrix).
    
    Note: We should also account for the final LayerNorm, but for simplicity
    we'll use the raw unembedding direction.
    """
    with torch.no_grad():
        # lm_head.weight has shape [vocab_size, hidden_size]
        direction = model.lm_head.weight[token_id].clone()
        # Normalize to unit vector
        direction = direction / direction.norm()
    return direction


def train_probe_direction(model, tokenizer, train_data, target_token_id, 
                          n_samples=5000, max_length=256):
    """
    Train a logistic regression probe to find the direction that predicts
    whether the next token is the target token.
    
    Returns the normalized probe direction.
    """
    from sklearn.linear_model import LogisticRegression
    
    model.eval()
    
    # Collect activations
    all_activations = []
    all_labels = []
    
    # Sample documents
    sample_indices = random.sample(range(len(train_data)), min(n_samples, len(train_data)))
    
    print(f"Collecting activations from {len(sample_indices)} documents...")
    for idx in tqdm(sample_indices):
        text = train_data[idx]['text']
        tokens = tokenizer(text, return_tensors='pt', truncation=True, 
                          max_length=max_length, padding=False)
        input_ids = tokens['input_ids'].to(device)
        
        if input_ids.shape[1] < 2:
            continue
        
        with torch.no_grad():
            outputs = model(input_ids=input_ids, output_hidden_states=True)
            # Get final layer hidden states (before LM head)
            final_hidden = outputs.hidden_states[-1]  # [1, seq_len, hidden]
            
            # For each position, label is whether NEXT token is target
            # Position i predicts token i+1
            activations = final_hidden[0, :-1, :].cpu().numpy()  # [seq_len-1, hidden]
            labels = (input_ids[0, 1:] == target_token_id).cpu().numpy()  # [seq_len-1]
            
            all_activations.append(activations)
            all_labels.append(labels)
    
    # Concatenate all
    X = np.concatenate(all_activations, axis=0)
    y = np.concatenate(all_labels, axis=0)
    
    print(f"Collected {len(X)} samples, {y.sum()} positive ({100*y.mean():.2f}%)")
    
    # Train logistic regression
    print("Training logistic regression probe...")
    clf = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0)
    clf.fit(X, y)
    
    print(f"Probe accuracy: {clf.score(X, y):.4f}")
    
    # Get direction (normalized)
    direction = torch.tensor(clf.coef_[0], dtype=torch.float32, device=device)
    direction = direction / direction.norm()
    
    return direction


# Get probe direction based on type
if probe_type == 'unembedding':
    print("Using unembedding direction...")
    probe_direction = get_unembedding_direction(model, target_token_id)
    print(f"Probe direction shape: {probe_direction.shape}")
    print(f"Probe direction norm: {probe_direction.norm().item():.4f}")
else:
    print("Training probe direction...")
    probe_direction = train_probe_direction(
        model, tokenizer, data['train_data'], target_token_id
    )
    print(f"Trained probe direction shape: {probe_direction.shape}")

In [ ]:
# Verify probe direction: compute correlation with actual logits
print("Verifying probe direction...")

# Test on a few documents
test_docs = random.sample(data['val_data'], 10)

correlations = []
for doc in test_docs:
    tokens = tokenizer(doc['text'], return_tensors='pt', truncation=True, 
                      max_length=max_length, padding=False)
    input_ids = tokens['input_ids'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=input_ids, output_hidden_states=True)
        final_hidden = outputs.hidden_states[-1][0]  # [seq_len, hidden]
        logits = outputs.logits[0]  # [seq_len, vocab]
        
        # Probe activation at each position
        probe_activation = final_hidden @ probe_direction  # [seq_len]
        
        # Actual logit for target token
        target_logit = logits[:, target_token_id]  # [seq_len]
        
        # Correlation
        corr = torch.corrcoef(torch.stack([probe_activation, target_logit]))[0, 1].item()
        correlations.append(corr)

print(f"Correlation between probe activation and target logit:")
print(f"  Mean: {np.mean(correlations):.4f}")
print(f"  Std: {np.std(correlations):.4f}")
print(f"  Range: [{min(correlations):.4f}, {max(correlations):.4f}]")

## Define Probe-Based Kronfluence Task

The measurement function computes how much the final residual stream aligns with the probe direction.

In [ ]:
from typing import Dict, List

BATCH_TYPE = Dict[str, torch.Tensor]


class ProbeBasedTask(Task):
    """
    Kronfluence task that measures alignment with a probe direction in the
    final residual stream.
    
    Measurement = sum of (probe_direction · final_residual) across positions
    
    This finds documents influential for how much the model's internal
    representations align with the target token direction.
    """
    
    def __init__(self, tokenizer, probe_direction: torch.Tensor, 
                 target_token_id: int, num_layers: int = 8):
        super().__init__()
        self.tokenizer = tokenizer
        self.probe_direction = probe_direction  # [hidden_dim], normalized
        self.target_token_id = target_token_id
        self.num_layers = num_layers
    
    def compute_train_loss(self, batch: BATCH_TYPE, model: nn.Module, 
                           sample: bool = False) -> torch.Tensor:
        """Standard LM training loss (cross-entropy)."""
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()
        
        logits = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))
        labels = batch["labels"][..., 1:].contiguous()
        
        if not sample:
            return F.cross_entropy(logits, labels.view(-1), reduction="sum", ignore_index=-100)
        else:
            with torch.no_grad():
                probs = F.softmax(logits.detach(), dim=-1)
                sampled_labels = torch.multinomial(probs, num_samples=1).flatten()
                masks = labels.view(-1) == -100
                sampled_labels[masks] = -100
            return F.cross_entropy(logits, sampled_labels, ignore_index=-100, reduction="sum")
    
    def compute_measurement(self, batch: BATCH_TYPE, model: nn.Module) -> torch.Tensor:
        """
        Measure alignment of final residual stream with probe direction.
        
        Returns sum of probe activations across all non-padding positions.
        """
        # Forward pass with hidden states
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            output_hidden_states=True,
        )
        
        # Get final layer hidden states (before LM head)
        # Shape: [batch_size, seq_len, hidden_dim]
        final_hidden = outputs.hidden_states[-1].float()
        
        # Project onto probe direction
        # probe_direction: [hidden_dim]
        # Result: [batch_size, seq_len]
        probe_activation = torch.einsum('bsh,h->bs', final_hidden, self.probe_direction)
        
        # Mask out padding positions
        attention_mask = batch["attention_mask"].float()
        probe_activation = probe_activation * attention_mask
        
        # Sum across all positions and batch
        return probe_activation.sum()
    
    def get_influence_tracked_modules(self) -> List[str]:
        """Modules to track for influence computation."""
        modules = []
        for i in range(self.num_layers):
            modules.extend([
                f"transformer.h.{i}.attn.attention.q_proj",
                f"transformer.h.{i}.attn.attention.k_proj",
                f"transformer.h.{i}.attn.attention.v_proj",
                f"transformer.h.{i}.attn.attention.out_proj",
                f"transformer.h.{i}.mlp.c_fc",
                f"transformer.h.{i}.mlp.c_proj",
            ])
        return modules
    
    def get_attention_mask(self, batch: BATCH_TYPE) -> torch.Tensor:
        return batch["attention_mask"]


# Create task
task = ProbeBasedTask(
    tokenizer=tokenizer,
    probe_direction=probe_direction,
    target_token_id=target_token_id,
    num_layers=model.config.num_layers
)

print(f"Created ProbeBasedTask for '{target_word}'")
print(f"Tracked modules: {len(task.get_influence_tracked_modules())}")

## Create Datasets

In [ ]:
class TextDataset(Dataset):
    """PyTorch Dataset for tokenized text data."""
    
    def __init__(self, data_list, tokenizer, max_length):
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        text = self.data[idx]['text']
        tokenized = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        input_ids = tokenized['input_ids'].squeeze(0)
        attention_mask = tokenized['attention_mask'].squeeze(0)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
        }


# Create training dataset
train_dataset = TextDataset(data['train_data'], tokenizer, max_length)
print(f"Training dataset: {len(train_dataset)} samples")

# Create query dataset (validation data)
# We use validation data as queries to find which training docs influence
# the probe activation on validation examples
query_dataset = TextDataset(data['val_data'][:500], tokenizer, max_length)  # Use subset
print(f"Query dataset: {len(query_dataset)} samples")

## Setup Kronfluence Analyzer

In [ ]:
# Prepare model for Kronfluence
model = prepare_model(model, task)

# Setup analyzer
analyzer = Analyzer(
    analysis_name=f"probe_{target_word.strip()}",
    model=model,
    task=task,
)

dataloader_kwargs = DataLoaderKwargs(
    num_workers=0,
    collate_fn=default_data_collator,
    pin_memory=True
)
analyzer.set_dataloader_kwargs(dataloader_kwargs)

print("Analyzer ready")

In [ ]:
# Fit factors (EK-FAC)
factors_name = f"ekfac_probe_{checkpoint}"

factor_args = extreme_reduce_memory_factor_arguments(
    strategy="ekfac", module_partitions=1, dtype=torch.bfloat16
)
factor_args.covariance_module_partitions = 2
factor_args.lambda_module_partitions = 4
factor_args.covariance_data_partitions = 4
factor_args.lambda_data_partitions = 4

analyzer.fit_all_factors(
    factors_name=factors_name,
    dataset=train_dataset,
    per_device_batch_size=4,
    factor_args=factor_args,
    overwrite_output_dir=False,
)

print("Factors fitted")

## Compute Influence Scores

In [ ]:
# Compute pairwise influence scores
score_args = ScoreArguments(damping_factor=damping)
print(f"Damping factor: {damping}")

analyzer.compute_pairwise_scores(
    scores_name="probe_influence_scores",
    factors_name=factors_name,
    query_dataset=query_dataset,
    train_dataset=train_dataset,
    per_device_query_batch_size=16,
    score_args=score_args,
    overwrite_output_dir=True,
)

print("Influence scores computed")

In [ ]:
# Load influence scores
scores = analyzer.load_pairwise_scores("probe_influence_scores")
influence_scores = scores["all_modules"]  # [n_queries, n_train]

print(f"Influence scores shape: {influence_scores.shape}")
print(f"  Queries: {influence_scores.shape[0]}")
print(f"  Training docs: {influence_scores.shape[1]}")

In [ ]:
# Aggregate influence scores across queries
mean_influence = influence_scores.mean(dim=0)  # [n_train]

print(f"\nInfluence score statistics:")
print(f"  Mean: {mean_influence.mean().item():.4f}")
print(f"  Std: {mean_influence.std().item():.4f}")
print(f"  Min: {mean_influence.min().item():.4f}")
print(f"  Max: {mean_influence.max().item():.4f}")

# Sort by influence
sorted_scores, sorted_indices = torch.sort(mean_influence)

## Analyze Most Influential Documents

- **Most negatively influential**: Documents that, if removed, would INCREASE probe activation. These docs currently "suppress" the target direction.
- **Most positively influential**: Documents that, if removed, would DECREASE probe activation. These docs currently "support" the target direction.

In [ ]:
# Get top influential documents
n_top = 50

# Most negatively influential (suppress the direction)
neg_indices = sorted_indices[:n_top]
neg_scores = sorted_scores[:n_top]

# Most positively influential (support the direction)
pos_indices = sorted_indices[-n_top:]
pos_scores = sorted_scores[-n_top:]

print(f"Most NEGATIVELY influential (suppress '{target_word}' direction):")
print(f"  Score range: {neg_scores[0].item():.2f} to {neg_scores[-1].item():.2f}")

print(f"\nMost POSITIVELY influential (support '{target_word}' direction):")
print(f"  Score range: {pos_scores[0].item():.2f} to {pos_scores[-1].item():.2f}")

In [ ]:
# Count target token occurrences in influential documents
def count_token_in_docs(indices, train_data, token_id, tokenizer):
    """Count occurrences of a token in documents."""
    counts = []
    for idx in indices:
        text = train_data[idx.item()]['text']
        tokens = tokenizer.encode(text, add_special_tokens=False)
        counts.append(tokens.count(token_id))
    return counts


neg_target_counts = count_token_in_docs(neg_indices, data['train_data'], target_token_id, tokenizer)
pos_target_counts = count_token_in_docs(pos_indices, data['train_data'], target_token_id, tokenizer)

print(f"Target token ('{target_word}') counts:")
print(f"\nMost NEGATIVELY influential docs:")
print(f"  Total: {sum(neg_target_counts)}")
print(f"  Mean per doc: {np.mean(neg_target_counts):.2f}")
print(f"  Docs with token: {sum(1 for c in neg_target_counts if c > 0)}/{n_top}")

print(f"\nMost POSITIVELY influential docs:")
print(f"  Total: {sum(pos_target_counts)}")
print(f"  Mean per doc: {np.mean(pos_target_counts):.2f}")
print(f"  Docs with token: {sum(1 for c in pos_target_counts if c > 0)}/{n_top}")

In [ ]:
# Visualize influence score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of all influence scores
axes[0].hist(mean_influence.numpy(), bins=100, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', alpha=0.7, label='Zero')
axes[0].set_xlabel('Influence Score')
axes[0].set_ylabel('Count')
axes[0].set_title(f"Distribution of Influence Scores for '{target_word}' Direction")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Bar chart comparing token counts
x = ['Negatively\nInfluential', 'Positively\nInfluential']
counts = [sum(neg_target_counts), sum(pos_target_counts)]
colors = ['#d62728', '#2ca02c']
bars = axes[1].bar(x, counts, color=colors, edgecolor='black')
axes[1].set_ylabel(f"Total '{target_word}' count")
axes[1].set_title(f"'{target_word}' Occurrences in Top {n_top} Influential Docs")
for bar, count in zip(bars, counts):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{count}', ha='center', va='bottom', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Show example documents
print("=" * 80)
print(f"MOST NEGATIVELY INFLUENTIAL DOCUMENTS (suppress '{target_word}')")
print("=" * 80)

for i in range(min(5, n_top)):
    idx = neg_indices[i].item()
    score = neg_scores[i].item()
    text = data['train_data'][idx]['text']
    count = neg_target_counts[i]
    
    print(f"\n--- Document #{i+1} (index={idx}, score={score:.2f}, '{target_word}' count={count}) ---")
    print(text[:500] + "..." if len(text) > 500 else text)

In [ ]:
print("=" * 80)
print(f"MOST POSITIVELY INFLUENTIAL DOCUMENTS (support '{target_word}')")
print("=" * 80)

for i in range(min(5, n_top)):
    idx = pos_indices[-(i+1)].item()  # Reverse order (most positive first)
    score = pos_scores[-(i+1)].item()
    text = data['train_data'][idx]['text']
    count = pos_target_counts[-(i+1)]
    
    print(f"\n--- Document #{i+1} (index={idx}, score={score:.2f}, '{target_word}' count={count}) ---")
    print(text[:500] + "..." if len(text) > 500 else text)

## Save Results

In [ ]:
import pickle

results = {
    'target_word': target_word,
    'target_token_id': target_token_id,
    'probe_type': probe_type,
    'probe_direction': probe_direction.cpu().numpy(),
    'mean_influence_scores': mean_influence.numpy(),
    'sorted_indices': sorted_indices.numpy(),
    'sorted_scores': sorted_scores.numpy(),
    'neg_indices': neg_indices.numpy(),
    'neg_scores': neg_scores.numpy(),
    'pos_indices': pos_indices.numpy(),
    'pos_scores': pos_scores.numpy(),
    'checkpoint': checkpoint,
}

save_path = f'/home/s5e/jrosser.s5e/infusion/probe_influence_{target_word.strip()}.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(results, f)

print(f"Results saved to {save_path}")

## Summary

This analysis found training documents most influential for the probe direction corresponding to the target token.

**Interpretation:**
- **Negatively influential** documents: When present in training, these documents cause the model's residual stream to point AWAY from the target direction. Removing them would increase probe activation.
- **Positively influential** documents: These documents train the model to have residual streams that align WITH the target direction. Removing them would decrease probe activation.

For infusion experiments, you would perturb the **negatively influential** documents to shift them toward supporting the target direction.